# 03 - 1D CNN with PyTorch
This notebook implements a 1D Convolutional Neural Network (CNN) for raw time-series classification.
We will:
1. Build a PyTorch `Dataset` to slice raw force signals from `data/raw/` into fixed-length windows.
2. Design a 1D-CNN capable of extracting deep frequency and amplitude features.
3. Train the model using a weighted Cross-Entropy loss (to handle class imbalance) and Early Stopping.
4. Evaluate the model using a Classification Report and Confusion Matrix.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### 1. PyTorch Dataset and DataLoader
The dataset reads CSV files directly from `data/raw/`. It chunks the continuous time-series (e.g. `Fz` cutting force) into fixed-length windows (e.g., 1024 samples) which act as the input tensors for the CNN.


In [ ]:
class CNCRawDataset(Dataset):
    def __init__(self, raw_csv_paths, sequence_length=1024):
        self.sequence_length = sequence_length
        self.samples = []
        self.labels = []
        
        print('Loading and slicing raw data files...')
        for path in raw_csv_paths:
            df = pd.read_csv(path)
            
            fz_col = 'Cutting Force - Fz (Newton)'
            if fz_col not in df.columns:
                continue
                
            signal = df[fz_col].values
            
            # Chunk the data into non-overlapping windows
            n_chunks = len(signal) // self.sequence_length
            
            for i in range(n_chunks):
                start_idx = i * self.sequence_length
                end_idx = start_idx + self.sequence_length
                
                chunk = signal[start_idx:end_idx]
                
                # Apply the same threshold logic to get the label
                fz_std = np.std(chunk)
                label = 1 if fz_std > 2.5 else 0
                
                # Reshape to [Channels, Sequence_Length] (1 channel)
                features = chunk.reshape(1, -1)
                
                self.samples.append(features)
                self.labels.append(label)
                
        if len(self.samples) > 0:
            self.samples = np.array(self.samples, dtype=np.float32)
            self.labels = np.array(self.labels, dtype=np.int64)
            print(f'Total windows generated: {len(self.samples)}')
        else:
            print('No valid data found to process.')
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.samples[idx])
        y = torch.tensor(self.labels[idx])
        return x, y

# Find all raw CSV files
raw_csv_files = glob.glob('../data/raw/*.csv')

if len(raw_csv_files) > 0:
    full_dataset = CNCRawDataset(raw_csv_files, sequence_length=1024)
    
    # Stratified Train/Val split
    labels = full_dataset.labels
    train_idx, val_idx = train_test_split(
        np.arange(len(labels)), test_size=0.2, random_state=42, stratify=labels
    )
    
    train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(full_dataset, val_idx)
    
    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
else:
    print('No raw CSV files found in ../data/raw/.')


### 2. 1D-CNN Architecture
A 3-block 1D Convolutional Neural Network designed for raw force signals.
It uses large kernel sizes initially to capture broad temporal patterns, and scales down.
Global Average Pooling is used before the classifier to robustly summarize temporal features across the window.


In [ ]:
class ChatterCNN1D(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super(ChatterCNN1D, self).__init__()
        
        # 1st Convolutional Block (Large kernel to capture broad temporal patterns)
        self.conv1 = nn.Conv1d(in_channels, 32, kernel_size=15, stride=1, padding=7)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # 2nd Convolutional Block
        self.conv2 = nn.Conv1d(32, 64, kernel_size=9, stride=1, padding=4)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # 3rd Convolutional Block
        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=4, stride=4)
        
        # Global Average Pooling (Aggregates features temporally)
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Fully Connected Classifier
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x shape: [batch_size, in_channels, sequence_length]
        
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        
        x = self.gap(x)
        x = x.view(x.size(0), -1) # Flatten
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

model = ChatterCNN1D(in_channels=1, num_classes=2).to(device)
print(model)

# Test architecture with a dummy input
dummy_x = torch.randn(16, 1, 1024).to(device) # Batch of 16, 1 channel, length 1024
dummy_out = model(dummy_x)
print('\nOutput shape for dummy batch [16, 1, 1024]:', dummy_out.shape)

### 3. Training Loop with Weighted Cross-Entropy & Early Stopping
To manage the class imbalance (Chatter is typically the minority class), we apply class weights to the Cross-Entropy loss.


In [ ]:
# We use a Weighted Cross-Entropy Loss to handle the positive class imbalance (~17% positive)
# The ratio of negative to positive is approx 5:1
weights = torch.tensor([1.0, 5.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Early stopping parameters
patience = 5
best_val_loss = float('inf')
patience_counter = 0
num_epochs = 50

train_losses = []
val_losses = []

# Note: This loop will only execute if raw data was loaded
if 'train_loader' in locals():
    print('Starting training...')
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * X_batch.size(0)
            
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                val_outputs = model(X_val)
                val_loss = criterion(val_outputs, y_val)
                running_val_loss += val_loss.item() * X_val.size(0)
                
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')
        
        # Early Stopping check
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_1d_cnn_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered at epoch {epoch+1}.')
                break
                
    # Plot training curve
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss', color='#FFC55A')
    plt.plot(val_losses, label='Validation Loss', color='#00E5A0')
    plt.title('CNN Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Weighted Cross-Entropy Loss')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()
else:
    print('Skipping training loop as `train_loader` is not defined (No raw CSVs found).')

### 4. Evaluation on Validation Set


In [ ]:
if 'val_loader' in locals():
    # Load the best model weights
    model.load_state_dict(torch.load('best_1d_cnn_model.pth'))
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            outputs = model(X_val)
            
            # Extract class predictions
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_val.cpu().numpy())
            
    print('Classification Report (Validation Set):\n')
    print(classification_report(all_targets, all_preds, target_names=['Stable (0)', 'Chatter (1)']))
    
    # Confusion Matrix Plot
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Stable (0)', 'Chatter (1)'], 
                yticklabels=['Stable (0)', 'Chatter (1)'])
    plt.title('Confusion Matrix - 1D CNN')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()
else:
    print('Skipping evaluation as `val_loader` is not defined.')